In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [5]:
# Load datasets
df1 = pd.read_csv('/home/wahba/Documents/cicid/cicids2017/original/csv/TrafficLabelling /Wednesday-workingHours.pcap_ISCX.csv')
df2 = pd.read_csv('/home/wahba/Documents/cicid/CICFlowMeter/data/daily/2025-11-21_Flow.csv')
df3 = pd.read_csv('/home/wahba/Documents/nids3/tests/flow/flow_tuesday_flow_generation4.csv')

In [6]:
# Cleaning column names
col_names = {col: col.strip() for col in df1.columns}
df1.rename(columns=col_names, inplace=True)
col_names = {col: col.strip() for col in df2.columns}
df2.rename(columns=col_names, inplace=True)
col_names = {col: col.strip() for col in df3.columns}
df3.rename(columns=col_names, inplace=True)

del col_names

In [7]:
df1['Label'].value_counts()

Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64

In [8]:
# Filter out only DoS slowhttptest attacks
df1_dos = df1[df1['Label'] == 'DoS Slowhttptest']

In [9]:
df1_dos['flow_key'] = (
    df1_dos['Source IP'].astype(str) + ':' +
    df1_dos['Source Port'].astype(str) + '-' +
    df1_dos['Destination IP'].astype(str) + ':' +
    df1_dos['Destination Port'].astype(str) + '-' +
    df1_dos['Protocol'].astype(str)
    )
df2['flow_key'] = (
    df2['Src IP'].astype(str) + ':' +
    df2['Src Port'].astype(str) + '-' +
    df2['Dst IP'].astype(str) + ':' +
    df2['Dst Port'].astype(str) + '-' +
    df2['Protocol'].astype(str)
    )
df3['Protocol'] = df3['Protocol'].replace({'TCP': 6, 'UDP': 17, 'ICMP': 1}).astype(int)
df3['flow_key'] = (
    df3['Source IP'].astype(str) + ':' +
    df3['Source Port'].astype(str) + '-' +
    df3['Destination IP'].astype(str) + ':' +
    df3['Destination Port'].astype(str) + '-' +
    df3['Protocol'].astype(str)
    )

/tmp/ipykernel_166928/2829646784.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_dos['flow_key'] = (
/tmp/ipykernel_166928/2829646784.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df3['Protocol'] = df3['Protocol'].replace({'TCP': 6, 'UDP': 17, 'ICMP': 1}).astype(int)


In [10]:
# Get unique flow keys for slowhttptest from df1

slowhttptest_flow_key = df1_dos['flow_key'].unique()

In [ ]:
# Filter df2 and df3 for flows matching slowhttptest flow keys
df2_dos = df2[df2['flow_key'].isin(slowhttptest_flow_key)].reset_index(drop=True)
df3_dos = df3[df3['flow_key'].isin(slowhttptest_flow_key)].reset_index(drop=True)

# Find flow keys appeared in both dataset
df2_dos_and_df3_dos_matched_flow = set(df2_dos['flow_key']) & set(df3_dos['flow_key'])

In [50]:
# derive new df comprising flow_key appeared on both df
df2_dos_matched_df3 = df2_dos[df2_dos['flow_key'].isin(df2_dos_and_df3_dos_matched_flow)]
df3_dos_matched_df2 = df3_dos[df3_dos['flow_key'].isin(df2_dos_and_df3_dos_matched_flow)] 

# Comparing CICFLOWMETER captured flows with custom python generated flow

In [ ]:
# Comparing shapes
print("df2_dos.shape: ", df2_dos.shape)
print("df2_dos_matched_df3.shape: ",df2_dos_matched_df3.shape)
print("df3_dos.shape: ", df3_dos.shape)
print("df2_dos_matched_df3.shape:", df3_dos_matched_df2.shape)

df2_dos.shape:  (659, 85)
df2_dos_matched_df3.shape:  (397, 85)
df3_dos.shape:  (376, 58)
df2_dos_matched_df3.shape: (254, 58)


In [52]:
print(df2_dos_and_df3_dos_matched_flow)

{'172.16.0.1:34436-192.168.10.50:80-6', '172.16.0.1:33580-192.168.10.50:80-6', '172.16.0.1:33474-192.168.10.50:80-6', '172.16.0.1:33660-192.168.10.50:80-6', '172.16.0.1:34728-192.168.10.50:80-6', '172.16.0.1:33492-192.168.10.50:80-6', '172.16.0.1:33518-192.168.10.50:80-6', '172.16.0.1:33654-192.168.10.50:80-6', '172.16.0.1:33480-192.168.10.50:80-6', '172.16.0.1:34434-192.168.10.50:80-6', '172.16.0.1:33634-192.168.10.50:80-6', '172.16.0.1:33638-192.168.10.50:80-6', '172.16.0.1:33498-192.168.10.50:80-6', '172.16.0.1:33620-192.168.10.50:80-6', '172.16.0.1:33610-192.168.10.50:80-6', '172.16.0.1:33504-192.168.10.50:80-6', '172.16.0.1:35078-192.168.10.50:80-6', '172.16.0.1:33606-192.168.10.50:80-6', '172.16.0.1:33426-192.168.10.50:80-6', '172.16.0.1:35210-192.168.10.50:80-6', '172.16.0.1:33694-192.168.10.50:80-6', '172.16.0.1:34054-192.168.10.50:80-6', '172.16.0.1:34722-192.168.10.50:80-6', '172.16.0.1:33418-192.168.10.50:80-6', '172.16.0.1:33490-192.168.10.50:80-6', '172.16.0.1:35056-192.16

In [53]:
flow_key = '172.16.0.1:34436-192.168.10.50:80-6'

print(df2_dos[df2_dos['flow_key'] == flow_key].index)
print(df3_dos[df3_dos['flow_key'] == flow_key].index)

Index([81, 83], dtype='int64')
Index([99, 218], dtype='int64')


In [54]:
print(df2_dos.columns)
print(df3_dos.columns)

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Len

In [58]:
print(df2_dos.loc[[81, 83], ['Timestamp', 'Flow Duration']])
print(df3_dos.loc[[99, 218], ['Start Time', 'Flow Duration']])

                 Timestamp  Flow Duration
81  21/11/2025 05:19:33 PM      116405965
83  21/11/2025 05:22:22 PM            107
       Start Time  Flow Duration
99   1.763717e+09   65752.542257
218  1.763717e+09  169151.077032
